In [1]:
from vv.models.latent_diffusion_copy.vq_model import VQModel
import torch
from vv.data.CIFAR10_loader import CIFAR10Loader
import lightning as L
import os

In [9]:
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
print("OMP_NUM_THREADS:", os.environ["OMP_NUM_THREADS"])
print("MKL_NUM_THREADS:", os.environ["MKL_NUM_THREADS"])
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


OMP_NUM_THREADS: 16
MKL_NUM_THREADS: 16
CUDA_VISIBLE_DEVICES: 2,3


In [10]:
config = {
    "learning_rate": 1e-4,
    "embed_dim": 8,
    "n_embed": 16384,
    "ddconfig": {
        "double_z": False,
        "z_channels": 8,
        "resolution": 32,
        "in_channels": 3,
        "out_ch": 3,
        "ch": 128,
        "ch_mult": [1, 1, 2, 2, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [16],
        "dropout": 0.0
    },
    "lossconfig": {
        "target": "vv.models.latent_diffusion_copy.loss.VQLPIPSWithDiscriminator",
        "params": {
            "disc_conditional": False,
            "disc_in_channels": 3,
            "disc_start": 1,
            "disc_weight": 0.75,
            "disc_num_layers": 2,
            "codebook_weight": 1.0
        }
    }
}

In [11]:
model = VQModel(learning_rate=config["learning_rate"],
                ddconfig=config["ddconfig"],
                lossconfig=config["lossconfig"],
                n_embed=config["n_embed"],
                embed_dim=config["embed_dim"],
                )

making attention of type 'vanilla' with 128 in_channels
making attention of type 'vanilla' with 128 in_channels
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 8, 2, 2) = 32 dimensions.
making attention of type 'vanilla' with 512 in_channels
making attention of type 'vanilla' with 128 in_channels
making attention of type 'vanilla' with 128 in_channels
making attention of type 'vanilla' with 128 in_channels


./venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
./venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.


In [5]:
loader = CIFAR10Loader(batch_size=32)
loader.prepare_data()
loader.setup("fit")
train_dataloader = loader.train_dataloader()

x = next(iter(train_dataloader))
print(x[0].shape)
print(x[1])

torch.Size([32, 3, 32, 32])
tensor([9, 6, 2, 3, 0, 3, 2, 3, 0, 9, 3, 6, 2, 0, 6, 6, 7, 1, 5, 9, 9, 1, 9, 3,
        6, 7, 4, 6, 3, 8, 1, 0])


In [6]:
#y = model(x[0])

In [7]:
#print(y[0].shape)

In [8]:
# train model with validation
trainer = L.Trainer(default_root_dir="./DELETE_ME",
                    max_epochs=100,
                    enable_checkpointing=True)
trainer.fit(model=model,
            datamodule=loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4]

  | Name            | Type                     | Params | Mode 
---------------------------------------------------------------------
0 | encoder         | Encoder                  | 26.2 M | train
1 | decoder         | Decoder                  | 38.4 M | train
2 | loss            | VQLPIPSWithDiscriminator | 15.4 M | train
3 | quantize        | VectorQuantizer2         | 131 K  | train
4 | quant_conv      | Conv2d                   | 72     | train
5 | post_quant_conv | Con

lr_d 0.0001
lr_g 0.0001
Epoch 0:  24%|██▍       | 300/1250 [00:30<01:36,  9.82it/s, v_num=9]       

Exception in thread Thread-4:
Traceback (most recent call last):
  File "/path/to/localpython/python3.11/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "./venv/lib/python3.11/site-packages/tensorboard/summary/writer/event_file_writer.py", line 244, in run
    self._run()
  File "./venv/lib/python3.11/site-packages/tensorboard/summary/writer/event_file_writer.py", line 275, in _run
    self._record_writer.write(data)
  File "./venv/lib/python3.11/site-packages/tensorboard/summary/writer/record_writer.py", line 40, in write
    self._writer.write(header + header_crc + data + footer_crc)
  File "./venv/lib/python3.11/site-packages/tensorboard/compat/tensorflow_stub/io/gfile.py", line 775, in write
    self.fs.append(self.filename, file_content, self.binary_mode)
  File "./venv/lib/python3.11/site-packages/tensorboard/compat/tensorflow_stub/io/gfile.py", line 167, in append
    self._write(filename, file_content, "ab" if binary_mode else "a")
  File "./v

FileNotFoundError: [Errno 2] No such file or directory: b'DELETE_ME/lightning_logs/version_9/events.out.tfevents.1741960202.compgpu8.3488410.0'